In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Tải dữ liệu
df = pd.read_csv('DATA/dataset.csv')
df_severity = pd.read_csv('DATA/Symptom-severity.csv')

# 2. HÀM CHUẨN HÓA (Xóa bỏ mọi khoảng trắng và gạch dưới để so khớp tuyệt đối)
def clean_string(s):
    if not isinstance(s, str): return s
    return s.strip().lower().replace(' ', '').replace('_', '')   

# 3. Tạo từ điển trọng số chuẩn
# Key là chuỗi đã làm sạch, Value là trọng số
severity_dict = {}
for _, row in df_severity.iterrows():
    cleaned_key = clean_string(row['Symptom'])
    severity_dict[cleaned_key] = row['weight']

# 4. Xác định danh sách triệu chứng duy nhất từ dataset
symptom_cols = df.columns[1:]
all_raw_symptoms = set()
for col in symptom_cols:
    all_raw_symptoms.update(df[col].dropna().unique())
sorted_symptoms = sorted(list(all_raw_symptoms))

# 5. Tạo ma trận TRỌNG SỐ
X = pd.DataFrame(0, index=df.index, columns=sorted_symptoms)

for i, row in df.iterrows():
    # Lấy các triệu chứng trong hàng đó
    row_symptoms = row[symptom_cols].dropna().values
    for s in row_symptoms:
        if s in X.columns:
            # Làm sạch tên triệu chứng để tra cứu trọng số
            cleaned_s = clean_string(s)
            # Lấy trọng số, nếu không thấy thì mặc định là 1
            weight = severity_dict.get(cleaned_s, 1)
            X.at[i, s] = weight

# 6. Mã hóa tên Bệnh và kết hợp kết quả
df['Disease_code'], _ = pd.factorize(df['Disease'])
final_df = pd.concat([df[['Disease', 'Disease_code']], X], axis=1)

# 7. Lưu file
final_df.to_csv('transformed_dataset.csv')
print("Hoàn thành việc xử lý dataset")

Hoàn thành việc xử lý dataset
